# BoneAge AI — RSNA training on Kaggle

Trains the bone age regression model described in `docs/model.md` on GPU, then runs evaluation. See `ml/README.md` for what each script does; this notebook just wires them together for a Kaggle session.

**Before running, in the Kaggle notebook's right-hand "Notebook options" panel:**
1. **Accelerator** → GPU (T4 x2 or P100).
2. **Internet** → **On** (needed to download pretrained ImageNet weights for the backbone).
3. **Add input → Datasets**:
   - `kmader/rsna-bone-age` (the RSNA dataset — search for it by name).
   - Your own code, as a **second** dataset (see step-by-step instructions given alongside this notebook — upload the repo's `ml/` folder as a new private Kaggle Dataset, since this notebook needs to `import boneage`).

No real metric in `docs/model.md` should be updated from a notebook run until you've actually looked at the results below — don't copy numbers in blind.

## 1. Locate the two input datasets

Kaggle mounts every attached dataset under `/kaggle/input/<dataset-slug>/`. Run this first and **read the printed paths** — dataset slugs depend on what you named them when uploading, so the guessed defaults below may need adjusting.

In [ ]:
import os

print("Everything under /kaggle/input:")
for name in sorted(os.listdir("/kaggle/input")):
    print(" -", name)

In [ ]:
# EDIT THESE to match what you saw printed above.
RSNA_DATASET_DIR = "/kaggle/input/rsna-bone-age"       # the kmader/rsna-bone-age dataset
CODE_DATASET_DIR = "/kaggle/input/boneage-ai-ml"       # your uploaded ml/ folder

print("RSNA dataset contents:")
for root, dirs, files in os.walk(RSNA_DATASET_DIR):
    depth = root.replace(RSNA_DATASET_DIR, "").count(os.sep)
    if depth > 2:
        continue
    print(" " * depth, os.path.basename(root) or root, f"({len(files)} files)" if files else "")

print()
print("Code dataset contents:")
for root, dirs, files in os.walk(CODE_DATASET_DIR):
    depth = root.replace(CODE_DATASET_DIR, "").count(os.sep)
    if depth > 2:
        continue
    print(" " * depth, os.path.basename(root) or root, f"({len(files)} files)" if files else "")

## 2. Install the `ml` package and its dependencies

This assumes `CODE_DATASET_DIR` contains the repo's `ml/` folder directly (i.e. `CODE_DATASET_DIR/boneage/`, `CODE_DATASET_DIR/requirements.txt` exist — if your upload nested it differently, adjust `ML_ROOT` below).

In [ ]:
import subprocess
import sys

ML_ROOT = CODE_DATASET_DIR  # adjust if your dataset upload nested the ml/ folder differently

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", f"{ML_ROOT}/requirements.txt"],
    check=True,
)

sys.path.insert(0, ML_ROOT)

import boneage  # noqa: F401  (just confirms the import works)
print("boneage package importable from:", ML_ROOT)

## 3. Point the config at the Kaggle dataset paths

Each RSNA split is loaded as its own images-dir + labels-CSV pair, same as locally (see `docs/data.md#splits`). The exact folder names inside `kmader/rsna-bone-age` can vary slightly between dataset versions — use the listing from step 1 to fix any path below that doesn't match.

If this dataset version does **not** ship a separately labelled validation set (some mirrors only include train + an unlabelled test set), this cell falls back to carving an internal validation split out of the training pool with `boneage.splits.random_split_dataframe` (fixed seed, so it's reproducible) — clearly logged below so you know which path was taken.

In [ ]:
import yaml
from pathlib import Path

import pandas as pd

from boneage.splits import random_split_dataframe

WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(exist_ok=True)

# --- Best-guess paths for the kmader/rsna-bone-age layout — verify against step 1's listing. ---
train_images_dir = f"{RSNA_DATASET_DIR}/boneage-training-dataset/boneage-training-dataset"
train_csv_guess = f"{RSNA_DATASET_DIR}/boneage-training-dataset.csv"

val_images_dir_guess = f"{RSNA_DATASET_DIR}/boneage-validation-dataset/boneage-validation-dataset"
val_csv_guess = f"{RSNA_DATASET_DIR}/boneage-validation-dataset.csv"

test_images_dir_guess = f"{RSNA_DATASET_DIR}/boneage-test-dataset/boneage-test-dataset"
test_csv_guess = f"{RSNA_DATASET_DIR}/boneage-test-dataset.csv"

if Path(val_csv_guess).exists():
    val_images_dir, val_csv = val_images_dir_guess, val_csv_guess
    print("Using a separately labelled validation set:", val_csv)
else:
    print("No separately labelled validation CSV found at", val_csv_guess)
    print("Falling back to an internal split of the training pool (fixed seed=42).")
    full_train_df = pd.read_csv(train_csv_guess)
    train_df, val_df, _ = random_split_dataframe(full_train_df, val_frac=0.1, test_frac=0.0, seed=42)

    train_csv_guess = str(WORKING_DIR / "train_split.csv")
    val_csv = str(WORKING_DIR / "val_split.csv")
    train_df.to_csv(train_csv_guess, index=False)
    val_df.to_csv(val_csv, index=False)
    val_images_dir = train_images_dir  # same images dir, just a different row subset

if Path(test_csv_guess).exists():
    test_images_dir, test_csv = test_images_dir_guess, test_csv_guess
else:
    print("No labelled test CSV found — reusing the validation split for 'test' too,")
    print("so evaluate.py still runs; treat the 'test' numbers as informal in that case.")
    test_images_dir, test_csv = val_images_dir, val_csv

kaggle_config = {
    "seed": 42,
    "data": {
        "rsna": {
            "train": {"images_dir": train_images_dir, "labels_csv": train_csv_guess},
            "val": {"images_dir": val_images_dir, "labels_csv": val_csv},
            "test": {"images_dir": test_images_dir, "labels_csv": test_csv},
        },
        "column_aliases": {
            "id": ["id", "Case ID", "case_id"],
            "boneage": ["boneage", "Bone Age", "bone_age"],
            "male": ["male", "Sex", "sex"],
        },
        "image_extension": ".png",
    },
    "preprocessing": {
        "image_size": 224,
        "normalize_mean": [0.485, 0.456, 0.406],
        "normalize_std": [0.229, 0.224, 0.225],
    },
    "augmentation": {
        "rotation_degrees": 10,
        "translate_frac": 0.05,
        "zoom_range": [0.95, 1.05],
        "brightness": 0.2,
        "contrast": 0.2,
    },
    "dataloader": {"batch_size": 32, "num_workers": 2},
    "model": {
        "backbone": "resnet18",
        "pretrained": True,
        "dropout": 0.3,
        "hidden_dim": 128,
    },
    "training": {
        "epochs": 50,
        "learning_rate": 1e-4,
        "weight_decay": 1e-5,
        "lr_scheduler": "cosine",
        "early_stopping_patience": 5,
        "mixed_precision": True,
        "checkpoint_dir": str(WORKING_DIR / "checkpoints"),
        "log_every_n_steps": 1,
    },
    "evaluation": {
        "age_band_edges_months": [0, 60, 120, 180, 240],
        "within_months_threshold": 12,
        "output_dir": str(WORKING_DIR / "results"),
    },
    "inference": {"mc_dropout_passes": 20},
}

config_path = WORKING_DIR / "config_kaggle.yaml"
with open(config_path, "w") as f:
    yaml.safe_dump(kaggle_config, f, sort_keys=False)

print("Wrote config to", config_path)

## 4. Train

Everything (loss, optimizer, LR schedule, early stopping, checkpointing, mixed precision) comes from the config written above — see `docs/model.md#training-plan` and `boneage/training.py`. This can take a while; Kaggle sessions have a runtime limit, so `early_stopping_patience` is there to stop once validation MAE stops improving.

In [ ]:
import logging

from boneage.config import load_config
from boneage.training import train_model

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

config = load_config(config_path)
summary = train_model(config)

print(f"Best epoch: {summary.best_epoch}")
print(f"Best val MAE (months): {summary.best_val_mae:.3f}")
print(f"Stopped early: {summary.stopped_early}")
print(f"Checkpoint saved to: {summary.checkpoint_path}")

## 5. Evaluate

Computes MAE overall / by sex / by age band and % within the configured threshold on val and test, and saves a predicted-vs-true scatter plot — the numbers that will eventually go into `docs/model.md` and the Model performance screen. **These are real results from this run** — copy them into `docs/model.md` by hand afterwards (don't let a future session fabricate different ones).

In [ ]:
from boneage.dataloaders import build_dataloaders_from_pipeline_config
from boneage.evaluation import evaluate_model, load_checkpoint, save_results_json, save_scatter_plot

device = "cuda"
model, checkpoint_meta = load_checkpoint(summary.checkpoint_path, device=device)
_, val_loader, test_loader = build_dataloaders_from_pipeline_config(config)

for split, loader in [("val", val_loader), ("test", test_loader)]:
    result = evaluate_model(
        model,
        loader,
        split=split,
        age_band_edges_months=config.evaluation.age_band_edges_months,
        within_months_threshold=config.evaluation.within_months_threshold,
        device=device,
    )
    save_results_json(result, Path(config.evaluation.output_dir) / f"{split}_results.json")
    save_scatter_plot(result, Path(config.evaluation.output_dir) / f"{split}_scatter.png")
    print(f"{split}: n={result.n_samples} MAE overall={result.mae_overall:.3f} months, "
          f"% within {result.within_threshold_months:.0f} months={result.percent_within_threshold:.1f}%")
    print(f"  by sex: {result.mae_by_sex}")
    print(f"  by age band: {result.mae_by_age_band}")

## 6. Download the checkpoint + results

Everything under `/kaggle/working` (the checkpoint, the JSON results, the scatter plots) becomes this notebook's **Output** once you commit/save a version. On the notebook page: **Save Version → Save & Run All**, then open the finished version's **Output** tab to download `checkpoints/best_model.pt`, `results/val_results.json`, `results/test_results.json`, and the two scatter PNGs.

Copy those files back into your local `ml/checkpoints/` and `ml/results/` (both gitignored — see repo root `.gitignore`), and update `docs/model.md`'s metrics table with the real numbers from `*_results.json`.